# PARC2026 — π0.5 Group-aware Dataset Cheap Ablation V2

`47_group_aware_ablation_manifests.ipynb` で確立した **trajectory-group-aware fixed eval** を前提に、
V0/V1/V2を同一π0.5 / seed / optimizer stepでscreeningします。

**legacy `dataset_ablation_manifests_v1` は使用禁止**です。
このNotebookはgroup-aware manifest schema v2とTrajectory Leakage Gate PASSを確認できない限り、学習へ進みません。

training lossは候補絞り込み専用です。最終dataset選定には固定/simulator評価が必要です。


## Self-contained preflight
fresh Colab runtimeから直接実行できます。workspace・repo・dataset・group-aware manifests・training envまでこのNotebook内で準備します。


In [ ]:
from pathlib import Path
import json, os, platform, shutil, subprocess, sys
print('python:', sys.version)
print('platform:', platform.platform())
subprocess.run(['nvidia-smi'], check=True)
ROOT=Path('/content/parc2026')
for p in [ROOT, ROOT/'vendor', ROOT/'cache', ROOT/'datasets', ROOT/'outputs']:
    p.mkdir(parents=True, exist_ok=True)
REPO=ROOT/'py_AI'
if not (REPO/'.git').exists():
    subprocess.run(['git','clone','https://github.com/yu37330/py_AI.git',str(REPO)], check=True)
else:
    subprocess.run(['git','-C',str(REPO),'fetch','origin','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'checkout','main'], check=True)
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only','origin','main'], check=True)
PI05_DIR=REPO/'examples/pi05_libero_finetune'
GIT_SHA=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'], text=True).strip()
print('repo:', REPO)
print('git:', GIT_SHA)


## Python 3.10 / HF token
π0.5 trainingはLeRobot v0.4.4 + Python 3.10で固定します。PaliGemma access tokenは表示しません。


In [ ]:
if shutil.which('uv') is None:
    subprocess.run([sys.executable,'-m','pip','install','-q','uv'], check=True)
subprocess.run(['uv','python','install','3.10'], check=True)
PY310=subprocess.check_output(['uv','python','find','3.10'], text=True).strip()
from getpass import getpass
if not os.environ.get('HF_TOKEN'):
    try:
        from google.colab import userdata
        tok=userdata.get('HF_TOKEN')
    except Exception:
        tok=None
    os.environ['HF_TOKEN']=tok or getpass('HF token (PaliGemma access): ')
print('python3.10:', PY310)
print('HF_TOKEN: set (not displayed)')


## Training datasetを取得
運営combinedが無い場合はcompact `lerobot/libero_plus` v3をrevision pinして取得します。
学習に画像が必要なので、このNotebookではmeta/data/videosを取得します。


In [ ]:
subprocess.run([sys.executable,'-m','pip','install','-q','huggingface_hub>=0.30','pyarrow>=16','pandas>=2'], check=True)
os.environ.setdefault('HF_HUB_DISABLE_XET','1')
import pandas as pd
from huggingface_hub import HfApi, hf_hub_download
PUBLIC_DATASET='lerobot/libero_plus'
PUBLIC_ROOT=ROOT/'datasets'/'public_libero_plus_v3_train'
ORGANIZER_ROOT=ROOT/'datasets'/'libero_combined_20hz'
configured=os.environ.get('PI05_DATASET_ROOT')
def train_ready(p):
    return (p/'meta'/'info.json').exists() and any(p.glob('data/**/*.parquet')) and (p/'videos').exists()
if configured and train_ready(Path(configured)):
    DATASET_ROOT=Path(configured)
    DATASET_ID=os.environ.get('PI05_DATASET_REPO_ID','local/libero_combined_20hz')
    DATASET_REVISION=None
elif train_ready(ORGANIZER_ROOT):
    DATASET_ROOT=ORGANIZER_ROOT
    DATASET_ID='local/libero_combined_20hz'
    DATASET_REVISION=None
else:
    api=HfApi()
    ds=api.dataset_info(PUBLIC_DATASET)
    DATASET_REVISION=ds.sha
    files=api.list_repo_files(PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION)
    required=[f for f in files if f.startswith('meta/') or (f.startswith('data/') and f.endswith('.parquet')) or (f.startswith('videos/') and f.endswith('.mp4'))]
    print('public revision:', DATASET_REVISION, 'files:', len(required))
    for i,filename in enumerate(required,1):
        if i==1 or i%20==0 or i==len(required):
            print(f'download {i}/{len(required)}: {filename}')
        hf_hub_download(repo_id=PUBLIC_DATASET, repo_type='dataset', revision=DATASET_REVISION, filename=filename, local_dir=str(PUBLIC_ROOT))
    DATASET_ROOT=PUBLIC_ROOT
    DATASET_ID=PUBLIC_DATASET
print('dataset:', DATASET_ID, '@', DATASET_REVISION)
print('root:', DATASET_ROOT)


## Group-aware manifestsをself-containedで準備
Static Quality → trajectory fingerprint → group-aware manifest V2まで不足分を作ります。
旧episode単位manifestはtrajectory table生成のためだけに使い、学習manifestには絶対に使いません。


In [ ]:
STATIC_OUT=ROOT/'outputs'/'static_quality_v1'
METRICS=STATIC_OUT/'episode_quality_metrics.csv'
if not METRICS.exists():
    subprocess.run([sys.executable,str(REPO/'tools/data/static_quality_analyzer.py'),'--root',str(DATASET_ROOT),'--out',str(STATIC_OUT),'--smooth-window','5','--robust-z-threshold','5.0'], check=True)
LEGACY_MANIFESTS=ROOT/'outputs'/'dataset_ablation_manifests_v1'
if not (LEGACY_MANIFESTS/'run_matrix.json').exists():
    cmd=[sys.executable,str(REPO/'tools/data/build_dataset_ablation_manifests.py'),'--metrics-csv',str(METRICS),'--out',str(LEGACY_MANIFESTS),'--dataset-id',DATASET_ID,'--seed','20260830','--eval-per-task','2']
    if DATASET_REVISION:
        cmd += ['--dataset-revision',DATASET_REVISION]
    subprocess.run(cmd, check=True)
LEAK_V1=ROOT/'outputs'/'trajectory_group_leakage_v1'
GROUP_MEMBERS=LEAK_V1/'trajectory_group_members.csv'
if not GROUP_MEMBERS.exists():
    subprocess.run([sys.executable,str(REPO/'tools/data/check_trajectory_group_leakage.py'),'--root',str(DATASET_ROOT),'--manifests-dir',str(LEGACY_MANIFESTS),'--out',str(LEAK_V1),'--metrics-csv',str(METRICS),'--round-decimals','6'], check=True)
MANIFEST_OUT=ROOT/'outputs'/'dataset_ablation_manifests_v2_group_aware'
cmd=[sys.executable,str(REPO/'tools/data/build_group_aware_ablation_manifests.py'),'--metrics-csv',str(METRICS),'--trajectory-groups-csv',str(GROUP_MEMBERS),'--out',str(MANIFEST_OUT),'--dataset-id',DATASET_ID,'--seed','20260830','--eval-per-task','2']
if DATASET_REVISION:
    cmd += ['--dataset-revision',DATASET_REVISION]
subprocess.run(cmd, check=True)
matrix=json.loads((MANIFEST_OUT/'run_matrix.json').read_text())
eval_manifest=json.loads((MANIFEST_OUT/'FIXED_EVAL_HOLDOUT.json').read_text())
print('manifest dir:', MANIFEST_OUT)
print('group-aware:', matrix.get('group_aware'))
print('fixed eval:', matrix['fixed_eval']['episode_count'], 'episodes /', matrix['fixed_eval_group_count'], 'groups')
print('protected:', matrix['protected_episode_count'], f"({matrix['protected_fraction']:.2%})")
display(pd.DataFrame([{'variant':k,'episodes':v['episode_count'],'frames':v['frame_count'],'tasks':v['task_count']} for k,v in matrix['variants'].items()]))


## Mandatory Trajectory Leakage Gate
group-aware manifestsに対して実trajectory hashを再検証します。
**exact/action overlapとも0**で、schema v2 / group-aware=true / dataset identity一致を満たさない限り学習へ進みません。


In [ ]:
LEAK_V2=ROOT/'outputs'/'trajectory_group_leakage_v2_group_aware'
subprocess.run([sys.executable,str(REPO/'tools/data/check_trajectory_group_leakage.py'),'--root',str(DATASET_ROOT),'--manifests-dir',str(MANIFEST_OUT),'--out',str(LEAK_V2),'--metrics-csv',str(METRICS),'--round-decimals','6','--fail-on-exact-leakage'], check=True)
leak_report=json.loads((LEAK_V2/'trajectory_group_leakage_summary.json').read_text())
leak_df=pd.read_csv(LEAK_V2/'manifest_leakage_report.csv')
display(leak_df)
assert matrix.get('schema_version') == 2
assert matrix.get('group_aware') is True
assert eval_manifest.get('schema_version') == 2
assert eval_manifest.get('group_aware') is True
assert matrix.get('dataset_id') == DATASET_ID
assert eval_manifest.get('dataset_id') == DATASET_ID
if DATASET_REVISION:
    assert matrix.get('dataset_revision') == DATASET_REVISION
    assert eval_manifest.get('dataset_revision') == DATASET_REVISION
assert leak_report['trajectory_leakage_gate'] == 'PASS'
assert int(leak_df['exact_leakage_group_count'].sum()) == 0
assert int(leak_df['action_leakage_group_count'].sum()) == 0
if DATASET_ID == 'lerobot/libero_plus' and matrix['source_summary']['episode_count'] == 14347:
    assert matrix['protected_episode_count'] == 674
    expected={'V0_RAW':13673,'V1_MULTI_FLAG_PRUNED_EXPERIMENTAL':13579,'V1_ALL_REVIEW_PRUNED_EXPERIMENTAL':13301,'V2_SQRT_BALANCED_RAW':10758}
    for name,n in expected.items():
        assert matrix['variants'][name]['episode_count'] == n, (name,matrix['variants'][name]['episode_count'])
print('Group-aware Manifest Gate: PASS')
print('Trajectory Leakage Gate: PASS')
print('TRAINING MANIFEST SAFETY GATE: PASS')


## π0.5 training env setup
Leakage Gate PASS後にだけtraining環境を作ります。venv/model cache/checkpointはColab一時領域に置きます。


In [ ]:
TRAIN_DATA_ROOT=ROOT/'cache'/'pi05-ablation-group-aware-v2'
LEROBOT_ROOT=ROOT/'vendor'/'lerobot-pi05-ablation-v2'
TRAIN_DATA_ROOT.mkdir(parents=True, exist_ok=True)
setup_env=os.environ.copy()
setup_env.update({'PYTHON':PY310,'DATA_ROOT':str(TRAIN_DATA_ROOT),'LEROBOT_ROOT':str(LEROBOT_ROOT)})
subprocess.run(['bash','scripts/setup_train.sh'], cwd=PI05_DIR, env=setup_env, check=True)
print('training env: ready')


## Cheap ablation設定
同一model / seed / optimizer steps / effective batchで比較します。
長時間runを誤って開始しないよう、初期値は `RUN_ABLATIONS=False` です。


In [ ]:
ABLATION_BS=4
ABLATION_GA=8
ABLATION_STEPS=150
ABLATION_SEED=1000
RUN_ABLATIONS=False
print('effective batch:', ABLATION_BS*ABLATION_GA)
print('variants:', matrix['cheap_ablation_order'])
print('manifest schema:', matrix['schema_version'], 'group-aware:', matrix['group_aware'])
print('RUN_ABLATIONS:', RUN_ABLATIONS)


In [ ]:
RESULTS_DIR=ROOT/'outputs'/'pi05_dataset_ablation_v2_group_aware'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
result_rows=[]
if RUN_ABLATIONS:
    assert leak_report['trajectory_leakage_gate'] == 'PASS'
    assert matrix.get('schema_version') == 2 and matrix.get('group_aware') is True
    for variant in matrix['cheap_ablation_order']:
        manifest=MANIFEST_OUT/f'{variant}.json'
        manifest_obj=json.loads(manifest.read_text())
        assert manifest_obj.get('schema_version') == 2, manifest
        assert manifest_obj.get('group_aware') is True, manifest
        assert manifest_obj.get('dataset_id') == DATASET_ID, manifest
        if DATASET_REVISION:
            assert manifest_obj.get('dataset_revision') == DATASET_REVISION, manifest
        assert manifest_obj.get('protected_eval_group_count') == matrix['fixed_eval_group_count'], manifest
        assert manifest_obj.get('protected_episode_count') == matrix['protected_episode_count'], manifest
        run_name='colab_ga_v2_'+variant.lower()
        env=os.environ.copy()
        env.update({'DATA_ROOT':str(TRAIN_DATA_ROOT),'LEROBOT_ROOT':str(LEROBOT_ROOT),'ABLATION_MANIFEST':str(manifest),'PI05_DATASET_ROOT':str(DATASET_ROOT),'PI05_DATASET_REPO_ID':DATASET_ID,'PI05_VIDEO_BACKEND':'pyav','ABLATION_BS':str(ABLATION_BS),'ABLATION_GA':str(ABLATION_GA),'ABLATION_STEPS':str(ABLATION_STEPS),'ABLATION_SEED':str(ABLATION_SEED),'RUN_NAME':run_name,'HF_TOKEN':os.environ['HF_TOKEN']})
        print('\n=== RUN', variant, '===')
        subprocess.run(['bash','-lc','source env_train.sh && bash scripts/cheap_ablation_pi05.sh'], cwd=PI05_DIR, env=env, check=True)
        summary_path=TRAIN_DATA_ROOT/'pi05-ft-outputs'/run_name/'cheap_ablation_summary.json'
        row=json.loads(summary_path.read_text())
        row['group_aware_manifest_schema']=manifest_obj['schema_version']
        row['protected_episode_count']=manifest_obj['protected_episode_count']
        row['dataset_revision']=manifest_obj.get('dataset_revision')
        row['git_sha']=GIT_SHA
        result_rows.append(row)
        (RESULTS_DIR/f'{variant}.json').write_text(json.dumps(row,indent=2)+'\n')
    (RESULTS_DIR/'comparison.json').write_text(json.dumps(result_rows,indent=2)+'\n')
else:
    print('skip: Gate確認後、実行する時だけ RUN_ABLATIONS=True に変更')


## 結果比較
lossはscreening参考値です。wall time / peak VRAM / manifest / dataset revisionも同時に保存します。


In [ ]:
if (RESULTS_DIR/'comparison.json').exists():
    result_rows=json.loads((RESULTS_DIR/'comparison.json').read_text())
    df=pd.DataFrame(result_rows)
    cols=['variant','episode_count','frame_count','optimizer_steps','effective_batch','wall_sec','peak_vram_mib','final_logged_loss_best_effort','last20_logged_loss_mean_best_effort','protected_episode_count']
    display(df[[c for c in cols if c in df.columns]].sort_values('last20_logged_loss_mean_best_effort'))
    print('SCREENING COMPLETE — do not promote from loss alone')
else:
    print('no comparison.json yet')


## Exit criteria / 次
4 variantのsummaryが揃ったら、lossだけでwinnerを決めず **V0 + 有望なclean/balance候補** を固定local/simulator evalへ送ります。
その後Model Selection Laneと同期します。
